# Shoryuken Scanner

Run cell-by-cell. If a cell errors, fix and re-run just that cell — no need to restart from the top.

## 1. Install dependencies
Run this once per environment (safe to skip if already installed).

In [1]:
import sys
!{sys.executable} -m pip install -q pandas numpy requests yfinance tqdm lxml

## 2. Point Jupyter at the project folder
Edit `PROJECT_DIR` to wherever you unzipped the `shoryuken` folder.

In [2]:
import sys
from pathlib import Path

PROJECT_DIR = Path("C:\\Jupyter_project\\shoryuken_project\\shoryuken")  # <-- change if needed

assert PROJECT_DIR.exists(), f"Can't find {PROJECT_DIR.resolve()} - fix the path above"

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Using project at:", PROJECT_DIR.resolve())

Using project at: C:\Jupyter_project\shoryuken_project\shoryuken


## 3. Imports
Autoreload means you can edit the `.py` files and re-run cells without restarting the kernel.

In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from data import DataLoader
from scanner import Scanner
from quality import PatternQuality
from report import Report

## 4a. (Recommended first) Diagnose the data source
Yahoo Finance has been aggressively rate-limiting/blocking `yfinance` since late 2024, and it's an ongoing, external issue - not something in this codebase. When it happens, downloads fail almost instantly (empty dataframe or an error like `YFRateLimitError`/`JSONDecodeError`) instead of timing out slowly.

Run this cell first. If it errors or returns nothing, the fixes are (roughly in order of how often they help):
1. `!{sys.executable} -m pip install -U yfinance` and restart the kernel - Yahoo changes its API often and yfinance ships fixes for it regularly.
2. Wait a while (minutes to hours) and retry - these blocks are usually IP-based and temporary.
3. Try a different network/IP if you can.

`downloader.py` now prints the real error (instead of silently returning nothing) if every ticker fails, so check the notebook output for the actual exception.

In [4]:
import yfinance as yf

test = yf.download("AAPL", period="5d", progress=False)
print(test)

if test.empty:
    print("\n^ Empty result - this confirms the download layer (Yahoo/yfinance), not this project's code, is the problem right now.")
else:
    print("\nLooks good - downloads are working.")

Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2026-10-19  342.940002  346.000000  340.119995  340.779999  34573200
2026-10-20  339.500000  343.059998  338.700012  339.859985  35851400
2026-10-21  343.989990  345.250000  338.829987  339.359985  35614100
2026-10-22  348.220001  348.359985  342.279999  344.940002  46898900
2026-10-23  347.730011  351.730011  345.750000  349.119995  39496500

Looks good - downloads are working.


## 4. Load the universe
This downloads (and caches to `shoryuken/cache/`) OHLCV data and runs the full indicator pipeline on each ticker. First run is slow (network + S&P 500 size); later runs reuse the cache.

In [5]:
from _run_method_scan import load_cached_market
import indicators
from time import perf_counter

load_started = perf_counter()

loader = DataLoader()
REFRESH_PRICES = True  # False = fully offline scan of the existing cache
if REFRESH_PRICES:
    # COMBINED previously omitted SP600. Expand coverage without changing prices provider.
    from downloader import (get_sp500_tickers, get_sp400_tickers,
                            get_sp600_tickers, get_nasdaq100_tickers,
                            download_universe)
    expected_tickers = set().union(get_sp500_tickers(), get_sp400_tickers(),
                                   get_sp600_tickers(), get_nasdaq100_tickers(),
                                   {"SPY", "QQQ", "IWM"})
    refreshed_prices = download_universe(sorted(expected_tickers))
prices_finished = perf_counter()
# Both scans and the gate use the same complete cached stock universe.
market, discovery_asof = load_cached_market(PROJECT_DIR / "cache" / "market_data.db")
cache_finished = perf_counter()
if REFRESH_PRICES:
    missing_prices = sorted(expected_tickers - set(market))
    stale_prices = sorted(t for t in expected_tickers & set(market)
                         if pd.Timestamp(market[t].index.max()).date() != pd.Timestamp(discovery_asof).date())
    print(f"Universe coverage: {len(expected_tickers)} expected; "
          f"{len(expected_tickers & set(market))} cached; "
          f"{len(missing_prices)} missing; {len(stale_prices)} not current")
    print("Missing price symbols:", missing_prices)
    print("Non-current price symbols:", stale_prices)
print("Discovery database:", (PROJECT_DIR / "cache" / "market_data.db").resolve())
indicator_errors = {}
for ticker, frame in market.items():
    try:
        indicators.compute_all(frame)
    except (ValueError, KeyError) as error:
        indicator_errors[ticker] = str(error)
indicators_finished = perf_counter()
print(f"Unified cached universe: {len(market)} tickers; data date {discovery_asof}")
print("Indicator errors:", indicator_errors)

# ETF histories are isolated from the stock discovery/Watchable universe.
from market_view import load_market_view
etf_started = perf_counter()
market_view_results = load_market_view(
    PROJECT_DIR / "cache" / "market_view", refresh=REFRESH_PRICES,
)
etfs_finished = perf_counter()
display(market_view_results)

print("Loading times (seconds):", {
    "price_refresh": round(prices_finished - load_started, 2),
    "cache_read": round(cache_finished - prices_finished, 2),
    "indicators": round(indicators_finished - cache_finished, 2),
    "etf_loading": round(etfs_finished - etf_started, 2),
})

# Cached membership snapshots; price provider remains Yahoo Finance.
from index_membership import load_index_memberships
index_memberships = load_index_memberships(
    PROJECT_DIR / "cache" / "index_membership", refresh=REFRESH_PRICES,
)
for key, snapshot in index_memberships.items():
    print(key, snapshot["Status"], len(snapshot["Tickers"]), snapshot.get("Error"))


Downloading:   0%|          | 0/1521 [00:00<?, ?ticker/s]

Failed to download BLFS: RuntimeError: Invalid OHLCV structure
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CWEN-A"}}}
$CWEN-A: possibly delisted; no timezone found

1 Failed download:
['CWEN-A']: possibly delisted; no timezone found
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
$CWEN-A: possibly delisted; no timezone found

1 Failed download:
['CWEN-A']: possibly delisted; no timezone found
$CWEN-A: possibly delisted; no timezone found

1 Failed download:
['CWEN-A']: possibly delisted; no timezone found
Failed to download CWEN-A: RuntimeError: Empty download (often Yahoo rate-limiting - see below)
$CSW: possibly delisted; no price data found  (1d 2026-10-17 -> 2026-10-24)

1 Failed download:
['CSW']: possibly delisted; no price data found  (1d 2026-10-17 -> 2026-10-24)
Failed to download HLX: RuntimeError: Invalid OHLCV structure
Failed to download LEG: 


Cache usage: 0 already fresh (no request sent), 1515 incremental (only new days fetched), 6 full history downloads.

6 of 1521 tickers failed to download.
Universe coverage: 1521 expected; 1515 cached; 6 missing; 1 not current
Missing price symbols: ['BLFS', 'CWEN-A', 'HLX', 'LEG', 'QRVO', 'VYLR']
Non-current price symbols: ['ACA']
Discovery database: C:\Jupyter_project\shoryuken_project\shoryuken\cache\market_data.db
Unified cached universe: 1527 tickers; data date 2026-10-23
Indicator errors: {'ADIG': 'Need at least 250 trading days.', 'FDXF': 'Need at least 250 trading days.', 'HONA': 'Need at least 250 trading days.', 'MBGL': 'Need at least 250 trading days.', 'MFP': 'Need at least 250 trading days.', 'SPCX': 'Need at least 250 trading days.', 'VGNT': 'Need at least 250 trading days.', 'VSNT': 'Need at least 250 trading days.'}


,Ticker,Name,Group,Source,PriceBasis,DownloadNote,FetchedUTC,Error,DataStatus,LastClose,...,ThreeMonthPct,SixMonthPct,YearPct,WeekStartDate,MonthStartDate,YTDPct,WeekRSPct,MonthRSPct,RSBenchmark,RSStatus
0,SPY,S&P 500,Market,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:53.788963+00:00,None,Downloaded,759.969971,...,3.080422,6.986557,15.029802,2026-10-16,2026-09-24,12.316088,NaN,NaN,SPY,Not a sector
1,QQQ,Nasdaq 100,Market,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:54.443345+00:00,None,Downloaded,704.400024,...,3.373894,6.330801,16.882141,2026-10-16,2026-09-24,15.055759,NaN,NaN,SPY,Not a sector
2,IWM,Russell 2000,Market,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:53.406491+00:00,None,Downloaded,275.750000,...,-5.612577,0.172668,14.470132,2026-10-16,2026-09-24,12.780560,NaN,NaN,SPY,Not a sector
3,RSP,S&P 500 Equal Weight,Market,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:54.049379+00:00,None,Downloaded,216.539993,...,1.010305,8.001649,15.887747,2026-10-16,2026-09-24,14.392286,NaN,NaN,SPY,Not a sector
4,QQQE,Nasdaq 100 Equal Weight,Market,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:54.716829+00:00,None,Downloaded,117.169998,...,0.889338,7.972279,14.658305,2026-10-16,2026-09-24,15.051398,NaN,NaN,SPY,Not a sector
5,XLK,Technology,Sector,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:55.026383+00:00,None,Downloaded,183.229996,...,5.246025,14.631479,29.020166,2026-10-16,2026-09-24,27.728634,0.300221,-5.003165,SPY,Available
6,XLF,Financials,Sector,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:55.276338+00:00,None,Downloaded,55.900002,...,-1.375103,9.479486,8.299617,2026-10-16,2026-09-24,3.311530,1.118994,3.484944,SPY,Available
7,XLE,Energy,Sector,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:55.540968+00:00,None,Downloaded,69.330002,...,19.503259,23.518667,62.429915,2026-10-16,2026-09-24,58.139179,2.035803,11.801524,SPY,Available
8,XLI,Industrials,Sector,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:55.944256+00:00,None,Downloaded,164.619995,...,-9.900415,-4.059526,8.917264,2026-10-16,2026-09-24,6.971046,-1.448200,-1.568570,SPY,Available
9,XLV,Health Care,Sector,Yahoo Finance,Yahoo auto-adjusted close,None,2026-10-24T16:20:56.244776+00:00,None,Downloaded,179.279999,...,10.138394,25.370950,24.880668,2026-10-16,2026-09-24,17.249951,1.892233,6.540812,SPY,Available


Loading times (seconds): {'price_refresh': 107.35, 'cache_read': 10.43, 'indicators': 113.03, 'etf_loading': 5.18}
SPY Cached 503 None
QQQ Cached 101 None
IWM Cached 1972 None


## 4b. Confirm the cache is actually being reused
The universe-loading cell above now prints a summary like `Cache usage: 480 already fresh, 20 incremental, 0 full history downloads` - that alone tells you whether it's working. If you ever see mostly "full history downloads" on a run where you expected caching, run the cell below to check where the database actually lives and what's in it.

Common cause if it's re-downloading everything every time: Jupyter's working directory changed between sessions (e.g. launched from a different shortcut/folder). The cache path shown below should be the same absolute path every time, regardless of where you launched Jupyter from - if it isn't, something's overriding it.

In [6]:
from cache import cache
import sqlite3

print('Cache database path:', cache.path)
print('File exists:', cache.path.exists())
print('File size (MB):', round(cache.path.stat().st_size / 1e6, 2) if cache.path.exists() else 0)

conn = sqlite3.connect(cache.path)
n_tickers = conn.execute('SELECT COUNT(DISTINCT ticker) FROM prices').fetchone()[0]
n_rows = conn.execute('SELECT COUNT(*) FROM prices').fetchone()[0]
date_range = conn.execute('SELECT MIN(date), MAX(date) FROM prices').fetchone()
conn.close()

print(f'Tickers cached: {n_tickers}')
print(f'Total rows: {n_rows}')
print(f'Date range: {date_range[0]} to {date_range[1]}')

Cache database path: C:\Jupyter_project\shoryuken_project\shoryuken\cache\market_data.db
File exists: True
File size (MB): 289.72
Tickers cached: 1527
Total rows: 3217737
Date range: 2018-01-02 to 2026-10-23


## 4c. Shared Watchable universe (run before all scans)
Rules: Stage 1–2 proxy, ATR14/Close ≥3%, average 20-session dollar volume ≥$50M, stock RS ≥70, sector rank ≥70, industry rank ≥70, market cap ≥$500M. Leadership ranks use the full cached universe before filtering. Missing metadata fails closed.
Market caps are dated current snapshots for live discovery, not historical point-in-time data. First run: set REFRESH_MARKET_CAPS=True to obtain a snapshot through yfinance, then return it to False to reuse it for up to seven days. Refresh can take several minutes. No price-cache or research outputs are modified.


In [7]:
import pandas as pd
from watchable import watchable_universe, ensure_market_caps, WatchableConfig
from _run_method_scan import load_cached_market

WATCHABLE_CONFIG = WatchableConfig()
WATCHABLE_OUTPUT_DIR = Path.home() / "Documents" / "Shoryuken_reports"
CAP_FILE = WATCHABLE_OUTPUT_DIR / "market_caps.csv"
if "market" not in globals():
    raise ValueError("Run unified universe section 4 first")
watchable_source_market = market
watchable_asof = discovery_asof
# Automatic: reuse fresh snapshots; refresh only missing/invalid/expired caps.
market_caps = ensure_market_caps(
    set(watchable_source_market) - {"SPY", "QQQ", "IWM"}, CAP_FILE,
    max_age_days=WATCHABLE_CONFIG.max_cap_age_days,
)
watchable_market, watchable_audit, watchable_summary = watchable_universe(
    watchable_source_market, market_caps, PROJECT_DIR / "data" / "sector_industry.csv",
    asof=watchable_asof, cfg=WATCHABLE_CONFIG,
)
watchable_tickers = set(watchable_market)
watchable_leadership = tuple(watchable_audit.set_index("Ticker")[column].dropna().to_dict() if column in watchable_audit else {} for column in ("RS", "SectorRank", "IndustryRank"))
WATCHABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
watchable_audit.to_csv(WATCHABLE_OUTPUT_DIR / "watchable_audit.csv", index=False)
print(watchable_summary)
display(pd.DataFrame(watchable_summary["funnel"]))
print("Exclusion counts (a stock can fail multiple rules):")
print(watchable_audit.ExclusionReasons.str.split("; ").explode().loc[lambda x: x.ne("")].value_counts())
display(watchable_audit.loc[watchable_audit.Watchable])
if not watchable_tickers:
    print("No stocks passed. Inspect watchable_audit.csv, especially missing/stale market caps and classifications. No thresholds are relaxed.")


Market caps: 1524 fresh; 0 need refresh
{'price_asof': '2026-10-23', 'metadata_report_date': '2026-10-24', 'universe_stocks': 1524, 'watchable': 79, 'excluded': 1445, 'ranking_universe': 'Full valid universe before filtering', 'market_cap_note': 'Current dated snapshot, not historical point-in-time data', 'funnel': [{'rule': 'PassHistory', 'before': 1524, 'remaining': 1499, 'removed_here': 25}, {'rule': 'PassStage', 'before': 1499, 'remaining': 842, 'removed_here': 657}, {'rule': 'PassATR', 'before': 842, 'remaining': 411, 'removed_here': 431}, {'rule': 'PassDollarVolume', 'before': 411, 'remaining': 296, 'removed_here': 115}, {'rule': 'PassRS', 'before': 296, 'remaining': 154, 'removed_here': 142}, {'rule': 'PassSector', 'before': 154, 'remaining': 82, 'removed_here': 72}, {'rule': 'PassIndustry', 'before': 82, 'remaining': 79, 'removed_here': 3}, {'rule': 'PassCapData', 'before': 79, 'remaining': 79, 'removed_here': 0}, {'rule': 'PassMarketCap', 'before': 79, 'remaining': 79, 'remove

,rule,before,remaining,removed_here
0,PassHistory,1524,1499,25
1,PassStage,1499,842,657
2,PassATR,842,411,431
3,PassDollarVolume,411,296,115
4,PassRS,296,154,142
5,PassSector,154,82,72
6,PassIndustry,82,79,3
7,PassCapData,79,79,0
8,PassMarketCap,79,79,0


Exclusion counts (a stock can fail multiple rules):
ExclusionReasons
Industry missing or rank below 70      1183
Sector missing or rank below 70        1084
Stock RS below 70                      1052
ATR below 3%                            740
Stage 1–2 proxy                         657
Dollar volume below $50M                476
Insufficient or stale price history      22
Market cap below $500M                    4
Invalid price/volume data                 3
Name: count, dtype: int64


,Ticker,AsOf,Watchable,PassHistory,PassStage,PassATR,PassDollarVolume,PassRS,PassSector,PassIndustry,...,ATRPct,AvgDollarVolume20,RS,Sector,Industry,SectorRank,IndustryRank,MarketCapUSD,MarketCapSnapshotDate,ExclusionReasons
24,ACN,2026-10-23,True,True,True,True,True,True,True,True,...,4.035929,1.781838e+09,91.0,Technology,Information Technology Services,81.0,83.0,1.405509e+11,2026-10-17,
30,ADI,2026-10-23,True,True,True,True,True,True,True,True,...,3.322610,1.289559e+09,74.0,Technology,Semiconductors,81.0,80.0,1.916166e+11,2026-10-17,
63,ALAB,2026-10-23,True,True,True,True,True,True,True,True,...,8.903141,1.309937e+09,81.0,Technology,Semiconductors,81.0,80.0,4.958551e+10,2026-10-17,
82,AMD,2026-10-23,True,True,True,True,True,True,True,True,...,5.293927,1.140368e+10,95.0,Technology,Semiconductors,81.0,80.0,8.631712e+11,2026-10-17,
106,APA,2026-10-23,True,True,True,True,True,True,True,True,...,3.193476,2.464629e+08,96.0,Energy,Oil & Gas E&P,99.0,91.0,1.721978e+10,2026-10-17,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1361,TWLO,2026-10-23,True,True,True,True,True,True,True,True,...,5.958518,1.226616e+09,97.0,Technology,Software - Infrastructure,81.0,94.0,3.752918e+10,2026-10-17,
1362,TXN,2026-10-23,True,True,True,True,True,True,True,True,...,3.222809,1.760535e+09,80.0,Technology,Semiconductors,81.0,80.0,2.524947e+11,2026-10-17,
1419,VLO,2026-10-23,True,True,True,True,True,True,True,True,...,4.016847,1.049995e+09,98.0,Energy,Oil & Gas Refining & Marketing,99.0,99.0,1.280442e+11,2026-10-17,
1453,WAT,2026-10-23,True,True,True,True,True,True,True,True,...,3.054290,3.925326e+08,88.0,Healthcare,Diagnostics & Research,90.0,96.0,4.290231e+10,2026-10-17,


## 5. Run the scan

In [8]:
quality = PatternQuality()
scanner = Scanner()

benchmark = market.get("SPY")  # same cached universe; no independent download
if benchmark is None:
    raise ValueError("SPY missing from unified cache; refresh prices before scanning")

# Market regime feeds into every result's ranking score now (not just position
# sizing) - QQQ/IWM need to be in your universe for this; if they're not
# (e.g. a small CUSTOM_TICKERS list), this just skips it and ranking falls
# back to a neutral regime component instead of crashing.
regime = None
qqq, iwm = market.get("QQQ"), market.get("IWM")
if qqq is not None and iwm is not None:
    from market_regime import MarketRegimeEngine
    regime = MarketRegimeEngine().analyze(benchmark, qqq, iwm)
    print(f"Market regime: {regime.trend} (score={regime.score:.0f}), distribution_days={regime.distribution_days}")
else:
    print("Market regime: skipped (QQQ/IWM not in this universe)")

assert "watchable_tickers" in globals(), "Run Watchable section 4c first"
results = scanner.scan(market, quality, benchmark=benchmark, regime=regime, candidate_tickers=watchable_tickers, leadership_snapshot=watchable_leadership, update_sector_data=False)
print(f"{len(results)} setups found")

Market regime: Bear (score=7), distribution_days=7


Scanning:   0%|          | 0/79 [00:00<?, ?ticker/s]

90 setups found


## 5a. (If 0 setups found) Diagnose why
Run this if `scan()` returns 0 results and you want to know whether that's genuinely quiet markets or something upstream broke. It shows, per ticker: whether a trend could even be computed, and every detector's raw score whether or not it cleared the detection threshold.

- **Most tickers show `TrendOK = False`** -> something's wrong upstream (bad/missing indicator columns, not enough history, etc.) - the tickers are getting skipped before any pattern is even checked.
- **`TrendOK = True` but scores are near 0 across the board** -> likely a real bug in a detector or the indicators feeding it.
- **Scores cluster a bit below each threshold** (e.g. mostly 50-65 against a 70 minimum) -> that's just a quiet day for setups, not a bug.

In [9]:
diag = scanner.diagnostics(market)

pattern_cols = [d.name for d in scanner.detectors]
print()
print(diag[pattern_cols].apply(lambda c: pd.to_numeric(c, errors='coerce')).describe())
if diag["DiagnosticError"].notna().any():
    display(diag.loc[diag.DiagnosticError.notna(), ["Ticker", "DiagnosticError"]])
if not diag.empty:
    scores = pd.to_numeric(diag[pattern_cols[0]], errors="coerce")
    display(diag.assign(_sort_score=scores).sort_values("_sort_score", ascending=False).drop(columns="_sort_score").head(20))

Thresholds (minimum score to count as detected): {'Shoryuken': 75, 'VCP': 70, 'DTRP': 70, 'PocketPivot': 70, 'FlatBase': 70, 'CupHandle': 70, 'EpisodicPivot': 70, 'DarvasBox': 70, 'HighTightFlag': 70, 'BaseOnBase': 70}
TrendOK: 1518 / 1527 tickers

         Shoryuken          VCP         DTRP  PocketPivot     FlatBase  \
count  1518.000000  1518.000000  1518.000000  1518.000000  1518.000000   
mean     27.976285    49.277997    27.899209    45.644269    35.361660   
std      16.134184     9.087456    17.889678    28.914644    18.234075   
min       0.000000    22.000000     0.000000     0.000000     0.000000   
25%      16.000000    42.000000    15.000000    20.000000    21.000000   
50%      25.000000    51.000000    21.000000    45.000000    37.000000   
75%      40.000000    56.000000    40.000000    66.000000    47.000000   
max      90.000000    86.000000   100.000000   111.000000    92.000000   

         CupHandle  EpisodicPivot    DarvasBox  HighTightFlag   BaseOnBase  
count  

,Ticker,DiagnosticError
31,ADIG,ValueError: Need at least 250 trading days.
528,FDXF,ValueError: Need at least 250 trading days.
659,HONA,ValueError: Need at least 250 trading days.
854,MBGL,ValueError: Need at least 250 trading days.
871,MFP,ValueError: Need at least 250 trading days.
1127,Q,ValueError: Need at least 250 trading days.
1262,SPCX,ValueError: Need at least 250 trading days.
1416,VGNT,ValueError: Need at least 250 trading days.
1441,VSNT,ValueError: Need at least 250 trading days.


,Ticker,TrendOK,DiagnosticError,Shoryuken,VCP,DTRP,PocketPivot,FlatBase,CupHandle,EpisodicPivot,DarvasBox,HighTightFlag,BaseOnBase
1354,TRV,True,None,90.0,71.0,55.0,106.0,81.0,76.0,51.0,86.0,56.0,70.0
146,AVTR,True,None,81.0,77.0,71.0,106.0,87.0,91.0,61.0,112.0,46.0,85.0
1447,VTRS,True,None,81.0,67.0,61.0,76.0,62.0,76.0,36.0,102.0,36.0,70.0
1100,PR,True,None,80.0,61.0,45.0,70.0,66.0,76.0,40.0,96.0,46.0,55.0
457,EIG,True,None,76.0,56.0,35.0,65.0,51.0,71.0,25.0,71.0,31.0,60.0
435,DVN,True,None,75.0,61.0,40.0,106.0,71.0,91.0,36.0,76.0,46.0,70.0
1244,SM,True,None,73.0,61.0,55.0,100.0,51.0,91.0,40.0,56.0,46.0,40.0
786,L,True,None,72.0,61.0,50.0,70.0,46.0,61.0,50.0,76.0,36.0,45.0
440,EA,True,None,72.0,75.0,65.0,76.0,50.0,60.0,36.0,70.0,30.0,80.0
1167,RNR,True,None,71.0,51.0,45.0,70.0,56.0,66.0,40.0,96.0,36.0,65.0


## 6. View + export the report
Score is the unchanged final ranking. Added fields show its component points, detector raw score, Rule Coverage (not success probability), prior-day setup metrics and signal-day breakout metrics. No new weighted score has been enabled.


In [9]:
report = Report.dataframe(results, market=market, asof=watchable_asof)
report.head(50)

,Ticker,Pattern,Score,RawDetectorScore,SelectionScoreBasis,Quality,Confidence,RS,SectorRank,IndustryRank,...,ScoreDiagnosticsStatus,DiagnosticsAsOf,SetupATRCompressionPct,SetupRangeCompressionPct,SetupVolumeDryUpRatio,SignalRVOLPrior20,SignalCloseLocationPct,Prior20HighReference,SignalDistancePriorHighPct,DiagnosticBasis
0,APA,CupHandle,84.49,76.0,Live composite,60.36,68.5,96.0,99.0,91.0,...,Available; reporting only,2026-10-23,2.722602,2.924553,1.059228,1.348094,82.562338,50.245618,3.292594,Setup excludes signal day; prior high is gener...
1,MPC,CupHandle,83.55,80.0,Live composite,51.79,72.1,99.0,99.0,99.0,...,Available; reporting only,2026-10-23,-7.319948,-10.250842,1.042148,0.728472,26.672559,493.980011,-7.441599,Setup excludes signal day; prior high is gener...
2,CHRD,CupHandle,83.30,81.0,Live composite,61.99,73.0,92.0,99.0,91.0,...,Available; reporting only,2026-10-23,-6.813270,-26.368677,1.248425,1.150804,87.814599,156.559998,1.002815,Setup excludes signal day; prior high is gener...
3,DINO,CupHandle,83.04,80.0,Live composite,51.36,72.1,98.0,99.0,99.0,...,Available; reporting only,2026-10-23,-1.521774,-8.801554,0.829609,0.878225,30.107659,129.369995,-5.959644,Setup excludes signal day; prior high is gener...
4,VLO,CupHandle,83.04,80.0,Live composite,51.36,72.1,98.0,99.0,99.0,...,Available; reporting only,2026-10-23,-9.055095,-17.496953,0.945058,0.943739,19.209043,461.519989,-9.152364,Setup excludes signal day; prior high is gener...
5,CRL,DarvasBox,82.98,96.0,Live composite,66.71,85.7,97.0,90.0,96.0,...,Available; reporting only,2026-10-23,-12.381803,-9.958049,1.441394,0.753847,87.824237,351.209991,-1.662822,Setup excludes signal day; prior high is gener...
6,APA,PocketPivot,82.30,96.0,Live composite,56.61,86.5,96.0,99.0,91.0,...,Available; reporting only,2026-10-23,2.722602,2.924553,1.059228,1.348094,82.562338,50.245618,3.292594,Setup excludes signal day; prior high is gener...
7,APA,DarvasBox,82.30,96.0,Live composite,56.61,85.7,96.0,99.0,91.0,...,Available; reporting only,2026-10-23,2.722602,2.924553,1.059228,1.348094,82.562338,50.245618,3.292594,Setup excludes signal day; prior high is gener...
8,RVTY,VCP,82.00,71.0,Live composite,59.39,55.9,96.0,90.0,96.0,...,Available; reporting only,2026-10-23,1.290486,15.271562,0.839068,0.601363,95.569645,166.250000,-4.499246,Setup excludes signal day; prior high is gener...
9,RVTY,CupHandle,82.00,85.0,Live composite,59.39,76.6,96.0,90.0,96.0,...,Available; reporting only,2026-10-23,1.290486,15.271562,0.839068,0.601363,95.569645,166.250000,-4.499246,Setup excludes signal day; prior high is gener...


In [10]:
report.to_csv("scan_results.csv", index=False)
print("Saved scan_results.csv")

Saved scan_results.csv


## 6a. Tagged method scans + entry-readiness report

Run section 4 and Watchable 4c first. All scans share the same cached universe and as-of date. REFRESH_PRICES=False in section 4 enables an offline run. The original Shoryuken scan/report can be skipped if only method results are wanted.

Methods: Qullamaggie breakout/base proxy, Minervini Trend Template (cross-sectional RS proxy, not official IBD RS), and Stockbee 4% daily / 20% weekly / 9M-share movers. A stock may carry several tags. These independent discovery screens do not change the original Shoryuken scan or its backtest.

Readiness is a reporting diagnostic, not trade approval. Ready = within 3% below a qualifying prior-base trigger; Breakout = above trigger with relative volume ≥1 and within extension limits; Extended = above trigger by >5% or >0.5 ATR; Forming = other valid matches; Unavailable = missing/stale/insufficient data. Inspect liquidity warnings and reasons. Invalidation is a base-low reference, not an execution stop. Earnings, sector and portfolio eligibility are not assessed.

Defaults are engineering proxies, not validated strategy rules. See `docs/READINESS_README.md` and `docs/METHOD_SCANS_README.md` for exact definitions. The report prints its data date; cached prices may be stale.


In [11]:
import pandas as pd
from IPython.display import display
from method_scans import scan_methods, summarize_tags, tag_existing_results, MethodScanConfig, METHODS
from readiness import readiness_report, ReadinessConfig
from _run_method_scan import load_cached_market

# "cache": full available history, read-only SQLite, no downloads.
# "market": reuse the market dictionary loaded in section 4.
METHOD_DATA_SOURCE = "cache"
METHOD_ASOF = None  # None = latest available date; or "2026-10-02"
METHOD_CONFIG = MethodScanConfig()
READINESS_CONFIG = ReadinessConfig()

if "watchable_source_market" not in globals():
    raise ValueError("Run unified universe section 4 and Watchable 4c first")
method_market = watchable_source_market
method_asof = watchable_asof
METHOD_DATA_SOURCE = "Unified cached universe"

assert "watchable_tickers" in globals(), "Run Watchable section 4c first"
if METHOD_ASOF is not None and pd.Timestamp(METHOD_ASOF).date() != pd.Timestamp(watchable_asof).date():
    raise ValueError("Current market-cap Watchable gate is live-only; historical method dates require point-in-time metadata.")
from qullamaggie_momentum import scan_qullamaggie_momentum
all_method_results = scan_methods(method_market, asof=method_asof, cfg=METHOD_CONFIG)
qull_results, qull_diagnostics = scan_qullamaggie_momentum(
    method_market, watchable_audit, method_asof)
other_results = all_method_results.loc[
    all_method_results["Method"].ne("Qullamaggie_Breakout") &
    all_method_results["Ticker"].isin(watchable_tickers)]
method_results = pd.concat([other_results, qull_results], ignore_index=True, sort=False)
method_results.attrs.update(all_method_results.attrs)
method_results.attrs["qullamaggie_rules"] = {
    "price_gt": 8, "adr20_gt": 4, "valid_cap_gte": 500_000_000,
    "liquidity": "daily dollar volume >20M OR average 20-day shares >300K",
    "trend": "SMA10>SMA20>SMA50; close above at least one",
    "roc": "21/63/126-session rankings; no cutoff", "confirmed_breakout_required": False}
qull_candidates = set(qull_results["Ticker"])
print("Qullamaggie momentum candidates:", len(qull_results))
print("Other methods retain the existing Watchable gate")
method_results.attrs["watchable_summary"] = watchable_summary
from dataclasses import asdict
method_results.attrs["watchable_config"] = asdict(WATCHABLE_CONFIG)
method_results.attrs["discovery_universe"] = sorted(watchable_source_market)
method_results.attrs["indicator_errors"] = indicator_errors
method_tags = summarize_tags(method_results)
entry_readiness = readiness_report(method_results, method_market, cfg=READINESS_CONFIG)

print("Data as of:", method_results.attrs.get("asof"), "| Source:", METHOD_DATA_SOURCE)
print("Eligible tickers:", method_results.attrs.get("eligible_tickers"))
print("Distinct matched tickers:", len(method_tags))
display(pd.DataFrame({"Method": METHODS, "Matches": [int((method_results.Method == m).sum()) for m in METHODS]}))
print("Readiness counts (not approved trades):")
display(entry_readiness.Readiness.value_counts().rename_axis("Readiness").to_frame("Tickers"))


Qullamaggie momentum candidates: 35
Other methods retain the existing Watchable gate
Data as of: 2026-10-23 | Source: Unified cached universe
Eligible tickers: 1508
Distinct matched tickers: 71


,Method,Matches
0,Qullamaggie_Breakout,35
1,Minervini_Trend_Template,39
2,Stockbee_4pct_Daily,11
3,Stockbee_20pct_Weekly,0
4,Stockbee_9M_Movers,1


Readiness counts (not approved trades):


,Tickers
Readiness,
Forming,64
Ready,5
Breakout,1
Extended,1


In [12]:
# Each matching method has its own tagged row; ticker_tags consolidates overlaps.
METHOD_TO_VIEW = "All"  # or any Method value from the table above
READINESS_TO_VIEW = "All"  # All / Ready / Breakout / Forming / Extended / Unavailable

method_view = method_results if METHOD_TO_VIEW == "All" else method_results.loc[method_results.Method == METHOD_TO_VIEW]
print("Tagged method matches:")
display(method_view)

readiness_view = entry_readiness if READINESS_TO_VIEW == "All" else entry_readiness.loc[entry_readiness.Readiness == READINESS_TO_VIEW]
print("Readiness, ranked within each status:")
display(readiness_view)


Tagged method matches:


,AsOf,Ticker,Method,Status,Close,DailyPct,WeeklyPct,Volume,AvgDollarVolume20,RVOL20,...,SMA10,SMA20,SMA50,ExclusionReasons,MonthPct,ThreeMonthPct,SixMonthPct,MonthRank,ThreeMonthRank,SixMonthRank
0,2026-10-23,CRWD,Minervini_Trend_Template,Trend qualified,257.570007,8.249981,13.272355,13367500.0,2.184098e+09,1.604992,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-10-23,PANW,Minervini_Trend_Template,Trend qualified,380.549988,7.221338,11.271926,6582700.0,2.044880e+09,1.248163,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-10-23,OKTA,Minervini_Trend_Template,Trend qualified,213.050003,5.821289,13.772300,3053200.0,6.721141e+08,0.935000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-10-23,MTDR,Minervini_Trend_Template,Trend qualified,59.959999,5.581965,4.242000,1922500.0,7.549492e+07,1.419207,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-10-23,CHRD,Minervini_Trend_Template,Trend qualified,158.130005,5.083738,2.309787,793600.0,1.010236e+08,1.150804,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,2026-10-23,RNG,Qullamaggie_Breakout,Momentum candidate,83.389999,4.998743,10.538172,4623100.0,1.293355e+08,3.106275,...,79.348000,78.290499,73.848600,,4.472563,55.605006,104.615452,591.0,16.0,20.0
82,2026-10-23,INSP,Qullamaggie_Breakout,Momentum candidate,71.110001,-2.133227,3.372583,592100.0,6.054051e+07,0.682627,...,70.366000,69.946500,67.433400,,3.342542,39.623012,26.283074,667.0,51.0,238.0
83,2026-10-23,MRNA,Qullamaggie_Breakout,Momentum candidate,200.100006,-1.248581,-8.215214,21524700.0,4.163416e+09,1.105918,...,222.831001,210.486002,171.146001,,2.710193,259.698009,294.441175,712.0,1.0,1.0
84,2026-10-23,FIVN,Qullamaggie_Breakout,Momentum candidate,37.610001,7.151004,16.692526,2468200.0,6.426463e+07,1.368098,...,34.714000,34.646000,33.946700,,2.591382,47.201572,127.250769,723.0,30.0,10.0


Readiness, ranked within each status:


,AsOf,Ticker,Tags,Statuses,Close,DailyPct,WeeklyPct,Volume,Readiness,TriggerPrice,...,VolumeDryUpRatio,LiquidityCheck,TradeEligibility,ReadinessReasons,RVOL20,ReadinessBaseDays,ReadinessBaseWidthPct,ContractionRatio,HigherLows,RankWithinStatus
0,2026-10-23,WAY,Qullamaggie_Breakout,Momentum candidate,28.850000,-0.448582,5.445905,1558600.0,Ready,29.250000,...,1.051701,Pass,Not assessed,Within diagnostic distance below prior trigger...,0.724384,10.0,12.629961,0.908866,True,1
1,2026-10-23,MANH,Minervini_Trend_Template,Trend qualified,238.190002,-1.643472,7.210696,928300.0,Ready,242.490005,...,1.120780,Pass,Not assessed,Within diagnostic distance below prior trigger...,1.618474,10.0,19.101178,0.927050,True,2
2,2026-10-23,WAT,Minervini_Trend_Template,Trend qualified,453.670013,-0.421428,3.826525,907600.0,Ready,465.359985,...,0.991048,Pass,Not assessed,Within diagnostic distance below prior trigger...,1.029714,30.0,15.723766,0.949436,True,3
3,2026-10-23,VSXY,Qullamaggie_Breakout,Momentum candidate,90.820000,2.714320,3.984433,1174200.0,Ready,93.389999,...,0.878433,Pass,Not assessed,Within diagnostic distance below prior trigger...,0.764642,10.0,14.574895,0.542256,True,4
4,2026-10-23,INSP,Qullamaggie_Breakout,Momentum candidate,71.110001,-2.133227,3.372583,592100.0,Ready,73.290001,...,0.787747,Pass,Not assessed,Within diagnostic distance below prior trigger...,0.682627,10.0,9.666316,0.861457,True,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,2026-10-23,UTHR,Qullamaggie_Breakout,Momentum candidate,587.500000,0.463418,10.200331,621300.0,Forming,NaN,...,0.638846,Pass,Not assessed,No qualifying prior contracting base; trigger ...,0.671077,NaN,NaN,NaN,NaN,61
67,2026-10-23,VLO,Minervini_Trend_Template,Trend qualified,419.279999,-1.642113,-5.718332,2402900.0,Forming,NaN,...,0.945058,Pass,Not assessed,No qualifying prior contracting base; trigger ...,0.943739,NaN,NaN,NaN,NaN,62
68,2026-10-23,WING,Qullamaggie_Breakout,Momentum candidate,119.540001,-1.100357,-9.246887,899600.0,Forming,NaN,...,1.014882,Pass,Not assessed,No qualifying prior contracting base; trigger ...,0.754685,NaN,NaN,NaN,NaN,63
69,2026-10-23,WS,Qullamaggie_Breakout,Momentum candidate,40.849998,0.938960,2.948584,513300.0,Forming,NaN,...,0.816967,Below existing liquidity floors,Not assessed,Below existing liquidity floors; discovery mat...,1.011259,NaN,NaN,NaN,NaN,64


In [13]:
import json
from dataclasses import asdict

METHOD_OUTPUT_DIR = PROJECT_DIR / "exports" / "method_scans"
METHOD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
qull_diagnostics.to_csv(METHOD_OUTPUT_DIR / "qullamaggie_diagnostics.csv", index=False)
for period in ["Month", "ThreeMonth", "SixMonth"]:
    qull_results.sort_values(period + "Rank").to_csv(
        METHOD_OUTPUT_DIR / ("qullamaggie_" + period.lower() + "_ranked.csv"), index=False)
method_results.to_csv(METHOD_OUTPUT_DIR / "method_results.csv", index=False)
method_tags.to_csv(METHOD_OUTPUT_DIR / "ticker_tags.csv", index=False)
entry_readiness.to_csv(METHOD_OUTPUT_DIR / "readiness_report.csv", index=False)
for method in METHODS:
    method_results.loc[method_results.Method == method].to_csv(METHOD_OUTPUT_DIR / (method + ".csv"), index=False)
# Write every status, including empty tables, so previous exports cannot linger.
for status in ["Ready", "Breakout", "Forming", "Extended", "Unavailable"]:
    entry_readiness.loc[entry_readiness.Readiness == status].to_csv(METHOD_OUTPUT_DIR / ("readiness_" + status.lower() + ".csv"), index=False)

metadata = dict(method_results.attrs)
metadata["data_source"] = METHOD_DATA_SOURCE
metadata["readiness_config"] = asdict(READINESS_CONFIG)
metadata["readiness_counts"] = entry_readiness.Readiness.value_counts().to_dict()
(METHOD_OUTPUT_DIR / "scan_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
# Keep the original scanner report separate and tag it if section 6 was run.
if "report" in globals() and isinstance(report, pd.DataFrame):
    tag_existing_results(report).to_csv(METHOD_OUTPUT_DIR / "shoryuken_tagged.csv", index=False)
print("Exported to:", METHOD_OUTPUT_DIR.resolve())


Exported to: C:\Jupyter_project\shoryuken_project\shoryuken\exports\method_scans


## 6b. Open the scan board in your browser
Run 6a first, plus sections 5–6 to include original Shoryuken results. This saves a standalone HTML file in your Documents/Shoryuken_reports folder, outside the Python project, and opens it in your default browser. The dark board groups matches by method and includes badges for every available stock field.


In [14]:
from scan_board import export_scan_board
import webbrowser

# Standalone output outside the Python project; change this folder if preferred.
HTML_OUTPUT_DIR = Path.home() / "Documents" / "Shoryuken_reports"
original_scan_report = report if "report" in globals() and isinstance(report, pd.DataFrame) else None
html_path = export_scan_board(
    method_results, entry_readiness, shoryuken=original_scan_report,
    market_view=market_view_results if "market_view_results" in globals() else None,
    market=method_market, breadth_market=market,
    index_memberships=index_memberships, market_caps=market_caps,
    sector_data=pd.read_csv(PROJECT_DIR / "data" / "sector_industry.csv"), watchable_audit=watchable_audit, history_dir=HTML_OUTPUT_DIR / "history", output=HTML_OUTPUT_DIR / "all_scan_results.html",
)
print("Saved standalone report:", html_path.resolve())
opened = webbrowser.open(html_path.resolve().as_uri(), new=2)
if not opened:
    print("Open the saved HTML file manually in your browser.")


Saved standalone report: C:\Users\biba0\Documents\Shoryuken_reports\all_scan_results.html


## 7. (Optional) Inspect a single ticker's setup
Useful for sanity-checking why something was or wasn't flagged.

In [ ]:
from patterns.shoryuken import Shoryuken

ticker = report.iloc[0]["Ticker"] if len(report) else next(iter(market))
df = market[ticker]

pattern = Shoryuken().detect(df)
print(ticker, "detected:", pattern.detected, "score:", round(pattern.score, 1))
for line in pattern.explanation:
    print(" ", line)

## 7b. Isolated scoring comparison (saved research artifact required)
This preserves the existing candidate set, exits, risk controls, full histories and stored/default research folds. It measures ranking changes only; new candidate signals require separate validation before any promotion.


In [ ]:
# Optional canonical fixed-candidate scoring comparison; never runs a new research scan.
from compare_scoring import compare_saved_candidates

ARTIFACT_PATH = PROJECT_DIR / "research_output" / "multiyear" / "candidate_trades.pkl"
COMPARISON_OUTPUT = Path.home() / "Documents" / "Shoryuken_reports" / "score_comparison"
if ARTIFACT_PATH.exists():
    comparison_summary = compare_saved_candidates(
        ARTIFACT_PATH, PROJECT_DIR / "cache" / "market_data.db",
        PROJECT_DIR / "data" / "sector_industry.csv", COMPARISON_OUTPUT,
    )
    print(comparison_summary)
    display(pd.read_csv(COMPARISON_OUTPUT / "fold_comparison.csv"))
else:
    print("Scoring validation blocked: original candidate_trades.pkl is absent. Set ARTIFACT_PATH to your saved canonical artifact. No production scoring change or new research scan has been performed.")


## 8. Backtest the screener (rigorous, no look-ahead)
This is a genuine walk-forward simulation, the way a trading platform's Strategy Tester works: at every historical bar, indicators are **recomputed from scratch** on a trailing window ending exactly at that bar - not reused from a single full-history calculation - so nothing can ever see future data. It answers one question: does this screening algorithm actually have predictive power?

**This is the expensive-but-honest version.** Recomputing indicators at every bar is slow across a large universe/long period. Use `step` (sample every Nth day) and a smaller universe while exploring; drop to `step=1` for a final validation run. `use_multiprocessing=True` parallelizes across tickers if you have multiple CPU cores.

**Known limitations (see `backtest.py`'s docstring for the full explanation):** survivorship bias (today's index membership, not point-in-time), simplified flat commission/slippage costs, and the equity curve doesn't model whether a position could actually have been filled at that size.

In [ ]:
from backtest import BacktestConfig, BacktestEngine

# Spec config: $100k, max 10 positions, equal-weight from cash, flat $30
# commission per side, next-open entries, and the single exit rule
# 'sell when Close < Entry - 1 x ATR(at entry)'. Override any field via
# keyword, e.g. BacktestConfig.spec(step=1) for a full-fidelity run.
cfg = BacktestConfig.spec(
    start_date="2023-01-01",
    end_date="2024-12-31",
    max_open_positions=10,
    atr_stop_multiple=1.0,
    step=1,               # every trading day
    lite_indicators=True, # verified identical detector outputs
    indicator_window=310,
    use_multiprocessing=False,
)

engine = BacktestEngine(scanner.detectors, cfg, sector_map=scanner.sector_map)

# Use full cached stock history; current market-cap snapshots are not historical.
backtest_market = {
    ticker: df for ticker, df in market.items()
    if ticker not in {"SPY", "QQQ", "IWM"} and len(df)
    and df.index.min() <= cfg.end_date
}
result = engine.run(backtest_market, benchmark=benchmark)

print(f"\n{len(result.signals)} signals, {len(result.trades)} candidate trades, "
      f"{len(result.accepted_trades)} taken by the {cfg.max_open_positions}-position portfolio\n")

# Full professional performance report (Overall / Risk / Trading / Portfolio):
result.print_report()

In [ ]:
print("Per-pattern breakdown:")
display(result.pattern_stats)

print("\nScore validation - does a higher score actually predict better results?")
display(result.score_buckets)

print("\nConfidence validation:")
display(result.confidence_buckets)

print("\nBy year:")
display(result.time_breakdown["by_year"])

print("\nBy market regime (needs the benchmark you loaded earlier):")
display(result.time_breakdown["by_regime"])

In [ ]:
result.equity_curve.plot(title="Equity Curve", figsize=(10, 4))

In [ ]:
# Writes to backtest_results/:
#   trade_log.csv              - every completed trade (spec columns)
#   daily_portfolio_value.csv  - daily equity / cash / utilization
#   equity_curve.png + drawdown, monthly heatmap, return/score charts
#   signals.csv, trades.csv, summary.csv, pattern_statistics.csv
out_dir = result.export("backtest_results")
print("Exported to:", out_dir)

In [ ]:
from cache import cache
df = cache.load("AAL")
if df is None:
    print("No cache entry for AAL at all — next run will attempt a full download.")
else:
    print("columns:", list(df.columns))
    print("dtypes:\n", df.dtypes)
    print("rows:", len(df))
    print("last date:", df.index[-1])
    print(df.tail(3))